# RAG Complete Tutorial - End-to-End Experimentation

This notebook brings everything together - chunking, embeddings, vector search, and LLM generation:

## What You'll Learn:
1. Complete RAG pipeline (retrieval + generation)
2. How chunking strategy affects RAG quality
3. How embedding model affects retrieval
4. How different LLMs perform
5. Comparing configurations side-by-side
6. Optimizing for your use case

## The Complete RAG Flow:
```
Documents → Chunking → Embeddings → Vector Store
                                         ↓
User Query → Embedding → Search → Retrieve Top-K
                                         ↓
          Context + Query → LLM → Answer + Sources
```

In [ ]:
import sys
sys.path.append('..')

from vectordb import (
    # Chunking
    chunk_text,
    ChunkingConfig,
    ChunkingStrategy,
    # Embeddings
    embed_chunks,
    get_provider,
    EmbeddingConfig,
    create_embed_function,
    # Vector Store
    get_backend,
    store_embeddings,
    VectorStoreConfig,
    # RAG
    rag_pipeline,
    RAGConfig,
    format_response_with_sources,
    retrieve_context
)
import pandas as pd
import time

## Part 1: Create Knowledge Base

Let's build a comprehensive knowledge base:

In [ ]:
knowledge_base = """
# Complete Machine Learning Guide

## What is Machine Learning?

Machine learning (ML) is a subset of artificial intelligence (AI) that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves.

The primary goal of machine learning is to enable computers to learn automatically without human intervention and adjust actions accordingly.

## Types of Machine Learning

### Supervised Learning

Supervised learning uses labeled training data to learn the relationship between input and output. The algorithm learns from examples with known answers.

Common supervised learning algorithms include:
- Linear Regression: Predicts continuous values
- Logistic Regression: Classification tasks
- Decision Trees: Tree-based decisions
- Random Forests: Ensemble of decision trees
- Support Vector Machines (SVM): Maximum margin classification
- Neural Networks: Deep learning models

### Unsupervised Learning

Unsupervised learning finds hidden patterns in unlabeled data. The algorithm must discover structure on its own.

Key unsupervised learning techniques:
- K-Means Clustering: Groups similar data points
- Hierarchical Clustering: Creates cluster trees
- Principal Component Analysis (PCA): Dimensionality reduction
- Autoencoders: Neural network-based compression

### Reinforcement Learning

Reinforcement learning trains agents to make decisions by rewarding desired behaviors and punishing undesired ones. The agent learns through trial and error.

Applications include robotics, game playing, and autonomous systems.

## Neural Networks and Deep Learning

Neural networks are computing systems inspired by biological neural networks. They consist of interconnected nodes (neurons) organized in layers.

### Deep Learning

Deep learning uses neural networks with multiple hidden layers to learn complex patterns. It has revolutionized fields like computer vision, natural language processing, and speech recognition.

Popular deep learning architectures:
- Convolutional Neural Networks (CNN): Image processing
- Recurrent Neural Networks (RNN): Sequential data
- Transformers: Attention-based models for NLP
- Generative Adversarial Networks (GANs): Generate new data

## ML Workflow

1. **Data Collection**: Gather relevant data
2. **Data Preprocessing**: Clean and prepare data
3. **Feature Engineering**: Extract meaningful features
4. **Model Selection**: Choose appropriate algorithm
5. **Training**: Fit model to training data
6. **Evaluation**: Test model performance
7. **Hyperparameter Tuning**: Optimize model parameters
8. **Deployment**: Deploy model to production

## Real-World Applications

Machine learning powers many modern applications:

**Healthcare**: Disease diagnosis, drug discovery, personalized treatment plans

**Finance**: Fraud detection, algorithmic trading, credit scoring, risk assessment

**E-commerce**: Product recommendations, customer segmentation, dynamic pricing

**Transportation**: Autonomous vehicles, traffic prediction, route optimization

**Natural Language Processing**: Chatbots, machine translation, sentiment analysis, text generation

**Computer Vision**: Face recognition, object detection, medical imaging analysis

## Challenges in Machine Learning

Common challenges include:
- Insufficient or poor quality training data
- Overfitting: Model memorizes training data
- Underfitting: Model too simple to capture patterns
- Bias in training data leading to unfair predictions
- Computational resources for training large models

## Best Practices

1. Start with simple models before trying complex ones
2. Always split data into training, validation, and test sets
3. Use cross-validation to assess model performance
4. Monitor for overfitting using validation data
5. Ensure diverse and representative training data
6. Document experiments and results
7. Consider ethical implications and fairness
"""

print(f"Knowledge base length: {len(knowledge_base)} characters")
print(f"Approximately {len(knowledge_base.split())} words")

## Part 2: Experiment 1 - Chunking Strategy Impact

Let's see how chunking affects RAG quality:

In [ ]:
# Test question
question = "What are the main supervised learning algorithms and what are they used for?"

# Test different chunking strategies
chunking_configs = [
    ("Small Fixed (256)", ChunkingConfig(strategy=ChunkingStrategy.FIXED, chunk_size=256, chunk_overlap=50)),
    ("Large Fixed (1024)", ChunkingConfig(strategy=ChunkingStrategy.FIXED, chunk_size=1024, chunk_overlap=100)),
    ("Small Recursive (256)", ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=256, chunk_overlap=50)),
    ("Large Recursive (1024)", ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=1024, chunk_overlap=100)),
]

print(f"Question: {question}\n")
print("="*80)
print("TESTING CHUNKING STRATEGIES")
print("="*80)

# Use simple embedding and vector store for all tests
embed_config = EmbeddingConfig(model="all-MiniLM-L6-v2")
provider = get_provider("sentence-transformers", embed_config)
embed_fn = create_embed_function("sentence-transformers", embed_config)

results_chunking = []

for name, chunk_config in chunking_configs:
    print(f"\nTesting: {name}")
    
    # Chunk
    chunks = chunk_text(knowledge_base, "ml_guide.md", chunk_config)
    print(f"  Created {len(chunks)} chunks")
    
    # Embed
    embeddings = embed_chunks(chunks, provider)
    
    # Store
    vector_config = VectorStoreConfig(collection_name=f"test_{name.replace(' ', '_')}")
    backend = get_backend("simple", vector_config)
    store_embeddings(embeddings, backend)
    
    # Retrieve (without LLM generation for now)
    retrieved = retrieve_context(question, backend, embed_fn, top_k=3)
    
    results_chunking.append({
        "Strategy": name,
        "Chunks": len(chunks),
        "Top Score": f"{retrieved[0].score:.3f}" if retrieved else "N/A",
        "Avg Chunk Size": f"{sum(len(c.content) for c in chunks) // len(chunks)} chars"
    })
    
    if retrieved:
        print(f"  Top match score: {retrieved[0].score:.3f}")
        print(f"  Content preview: {retrieved[0].chunk.content[:150]}...")

# Summary
df = pd.DataFrame(results_chunking)
print("\n\nSUMMARY - CHUNKING COMPARISON")
print("="*80)
print(df.to_string(index=False))

print("\n💡 Observations:")
print("- Smaller chunks: More granular, higher precision")
print("- Larger chunks: More context per retrieval")
print("- Recursive: Better preserves semantic boundaries")
print("- Fixed: Faster but may break context")

## Part 3: Experiment 2 - Embedding Model Impact

Compare different embedding models:

In [ ]:
# Use consistent chunking
chunk_config = ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=512, chunk_overlap=50)
chunks = chunk_text(knowledge_base, "ml_guide.md", chunk_config)

# Test different embedding models
embedding_models = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
]

print("="*80)
print("TESTING EMBEDDING MODELS")
print("="*80)

results_embeddings = []

for model_name in embedding_models:
    print(f"\nTesting: {model_name}")
    
    # Create provider
    embed_config = EmbeddingConfig(model=model_name)
    provider = get_provider("sentence-transformers", embed_config)
    embed_fn = create_embed_function("sentence-transformers", embed_config)
    
    # Embed
    start = time.time()
    embeddings = embed_chunks(chunks, provider)
    embed_time = time.time() - start
    
    # Store
    vector_config = VectorStoreConfig(collection_name=f"test_{model_name.replace('/', '_')}")
    backend = get_backend("simple", vector_config)
    store_embeddings(embeddings, backend)
    
    # Retrieve
    retrieved = retrieve_context(question, backend, embed_fn, top_k=3)
    
    results_embeddings.append({
        "Model": model_name,
        "Dimensions": provider.dimensions,
        "Embed Time (s)": f"{embed_time:.2f}",
        "Top Score": f"{retrieved[0].score:.3f}" if retrieved else "N/A"
    })
    
    if retrieved:
        print(f"  Dimensions: {provider.dimensions}")
        print(f"  Top score: {retrieved[0].score:.3f}")

# Summary
df = pd.DataFrame(results_embeddings)
print("\n\nSUMMARY - EMBEDDING MODEL COMPARISON")
print("="*80)
print(df.to_string(index=False))

print("\n💡 Observations:")
print("- Smaller models (384 dims): Faster embedding")
print("- Larger models (768+ dims): Better semantic understanding")
print("- all-MiniLM-L6-v2: Best default choice")
print("- all-mpnet-base-v2: Better quality if speed not critical")

## Part 4: Experiment 3 - top_k Impact on Context Quality

How many chunks should we retrieve?

In [ ]:
# Setup
embed_config = EmbeddingConfig(model="all-MiniLM-L6-v2")
provider = get_provider("sentence-transformers", embed_config)
embed_fn = create_embed_function("sentence-transformers", embed_config)

chunk_config = ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=512, chunk_overlap=50)
chunks = chunk_text(knowledge_base, "ml_guide.md", chunk_config)
embeddings = embed_chunks(chunks, provider)

vector_config = VectorStoreConfig(collection_name="test_topk")
backend = get_backend("simple", vector_config)
store_embeddings(embeddings, backend)

# Test different top_k values
print("="*80)
print("TESTING top_k VALUES")
print("="*80)

for k in [1, 3, 5, 10]:
    retrieved = retrieve_context(question, backend, embed_fn, top_k=k)
    
    total_chars = sum(len(r.chunk.content) for r in retrieved)
    avg_score = sum(r.score for r in retrieved) / len(retrieved) if retrieved else 0
    
    print(f"\ntop_k={k}:")
    print(f"  Chunks retrieved: {len(retrieved)}")
    print(f"  Total context: {total_chars} characters")
    print(f"  Average score: {avg_score:.3f}")
    print(f"  Score range: {retrieved[-1].score:.3f} to {retrieved[0].score:.3f}")

print("\n💡 Guidelines:")
print("  top_k=1-3: Precise, focused answers")
print("  top_k=5: Good balance (RECOMMENDED for RAG)")
print("  top_k=10+: More comprehensive but may include noise")

## Part 5: Complete RAG Pipeline with LLM

Now let's generate actual answers using an LLM:

**Note**: You need to configure an LLM provider (Ollama recommended for local/free)

In [ ]:
# Setup optimal configuration (based on experiments)
chunk_config = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,
    chunk_size=512,
    chunk_overlap=50
)

embed_config = EmbeddingConfig(model="all-MiniLM-L6-v2")
provider = get_provider("sentence-transformers", embed_config)
embed_fn = create_embed_function("sentence-transformers", embed_config)

# Chunk and embed
chunks = chunk_text(knowledge_base, "ml_guide.md", chunk_config)
embeddings = embed_chunks(chunks, provider)

# Store
vector_config = VectorStoreConfig(collection_name="ml_rag_demo")
backend = get_backend("simple", vector_config)
store_embeddings(embeddings, backend)

print("✓ Knowledge base ready for RAG!")
print(f"  Chunks: {len(chunks)}")
print(f"  Embeddings: {len(embeddings)}")
print(f"  Model: {embed_config.model}")

## Part 6: RAG Query with Ollama (Local, Free)

Make sure you have Ollama installed and a model pulled:
```bash
ollama pull llama3.2
```

In [ ]:
# Configure RAG
rag_config = RAGConfig(
    model="llama3.2",  # or "mistral", "phi", etc.
    temperature=0.7,
    max_tokens=500,
    top_k=5,
)

# Test queries
test_questions = [
    "What are the main supervised learning algorithms?",
    "Explain the difference between supervised and unsupervised learning.",
    "What are the challenges in machine learning?",
]

print("="*80)
print("RAG QUESTION ANSWERING")
print("="*80)

for question in test_questions:
    print(f"\n{'='*80}")
    print(f"Q: {question}")
    print(f"{'='*80}")
    
    try:
        # Run RAG pipeline
        response = rag_pipeline(
            question=question,
            backend=backend,
            embed_fn=embed_fn,
            provider_type="ollama",
            config=rag_config
        )
        
        print(f"\nA: {response.answer}")
        print(f"\nSources ({len(response.sources)} chunks):")
        for i, src in enumerate(response.sources, 1):
            print(f"  {i}. [{src.score:.3f}] {src.chunk.content[:80]}...")
    
    except Exception as e:
        print(f"\n⚠️ Error: {e}")
        print("\nTip: Make sure Ollama is running and you have a model pulled.")
        print("Install: https://ollama.ai/")
        print("Pull model: ollama pull llama3.2")
        break

## Part 7: Compare LLM Providers

Test different LLM providers (if you have API keys configured):

In [ ]:
question = "What is deep learning and how is it different from traditional machine learning?"

# Providers to test (uncomment if you have API keys)
providers_to_test = [
    ("ollama", RAGConfig(model="llama3.2", max_tokens=300)),
    # ("openai", RAGConfig(model="gpt-3.5-turbo", max_tokens=300)),
    # ("anthropic", RAGConfig(model="claude-3-5-sonnet-20241022", max_tokens=300)),
]

print(f"Question: {question}\n")

for provider_type, config in providers_to_test:
    print(f"\n{'='*80}")
    print(f"LLM: {provider_type} ({config.model})")
    print(f"{'='*80}")
    
    try:
        start = time.time()
        response = rag_pipeline(
            question=question,
            backend=backend,
            embed_fn=embed_fn,
            provider_type=provider_type,
            config=config
        )
        elapsed = time.time() - start
        
        print(f"\nAnswer: {response.answer}")
        print(f"\nMetadata:")
        print(f"  Time: {elapsed:.2f}s")
        print(f"  Tokens: {response.tokens_used if response.tokens_used else 'N/A'}")
        print(f"  Sources: {len(response.sources)}")
    
    except Exception as e:
        print(f"\n⚠️ Error: {e}")
        print(f"Skipping {provider_type}...")

## Part 8: RAG Without Retrieval (Comparison)

Let's see the difference between RAG and just asking the LLM:

In [ ]:
from vectordb.rag import get_llm_provider

question = "What are the best practices for machine learning mentioned in the guide?"

print("="*80)
print("WITH RAG (Using your documents)")
print("="*80)

try:
    response_with_rag = rag_pipeline(
        question=question,
        backend=backend,
        embed_fn=embed_fn,
        provider_type="ollama",
        config=RAGConfig(model="llama3.2", max_tokens=300, top_k=3)
    )
    
    print(f"\nAnswer: {response_with_rag.answer}")
    print(f"\nSources used: {len(response_with_rag.sources)} chunks")
    
    print("\n" + "="*80)
    print("WITHOUT RAG (General knowledge only)")
    print("="*80)
    
    provider = get_llm_provider("ollama", RAGConfig(model="llama3.2", max_tokens=300))
    answer_no_rag, _ = provider.generate(question)
    
    print(f"\nAnswer: {answer_no_rag}")
    
    print("\n" + "="*80)
    print("💡 KEY DIFFERENCES:")
    print("="*80)
    print("✅ WITH RAG: Uses YOUR specific document content")
    print("✅ WITH RAG: Can cite sources")
    print("✅ WITH RAG: More accurate for domain-specific questions")
    print("❌ WITHOUT RAG: Uses general knowledge only")
    print("❌ WITHOUT RAG: May hallucinate or provide generic answers")

except Exception as e:
    print(f"Error: {e}")
    print("Make sure Ollama is running: ollama serve")

## Part 9: Optimal Configuration Summary

Based on experiments, here are the recommended settings:

In [ ]:
print("OPTIMAL RAG CONFIGURATION")
print("="*80)

optimal_config = {
    "Chunking": {
        "Strategy": "RECURSIVE",
        "Chunk Size": "512 tokens",
        "Overlap": "50 tokens (10%)",
        "Reason": "Best balance of context and precision"
    },
    "Embeddings": {
        "Model": "all-MiniLM-L6-v2",
        "Dimensions": "384",
        "Provider": "sentence-transformers (local)",
        "Reason": "Fast, good quality, no API costs"
    },
    "Vector Store": {
        "Backend": "simple (dev) / pgvector (prod)",
        "Distance Metric": "cosine",
        "Reason": "Cosine best for normalized text embeddings"
    },
    "Retrieval": {
        "top_k": "5",
        "Min Score": "0.7 (highly relevant)",
        "Reason": "Good context without noise"
    },
    "LLM": {
        "Provider": "ollama (local) / openai (cloud)",
        "Model": "llama3.2 / gpt-3.5-turbo",
        "Temperature": "0.7",
        "Max Tokens": "500-1000",
        "Reason": "Balance between cost, speed, and quality"
    }
}

for category, settings in optimal_config.items():
    print(f"\n{category}:")
    for key, value in settings.items():
        print(f"  {key}: {value}")

## Part 10: Your Turn - Full RAG Experiment

Build your own RAG system:

In [ ]:
# YOUR CUSTOM RAG SYSTEM

# 1. Add your document
my_document = """
Paste your document here...
"""

# 2. Configure (adjust as needed)
my_chunk_config = ChunkingConfig(
    strategy=ChunkingStrategy.RECURSIVE,
    chunk_size=512,
    chunk_overlap=50
)

my_embed_config = EmbeddingConfig(model="all-MiniLM-L6-v2")

my_rag_config = RAGConfig(
    model="llama3.2",
    temperature=0.7,
    max_tokens=500,
    top_k=5
)

# 3. Build pipeline
# TODO: Add your pipeline code here

# 4. Ask questions
my_questions = [
    "Your question 1?",
    "Your question 2?",
]

# TODO: Query your RAG system

## Summary

### What You Learned:

✅ **Complete RAG pipeline**: From documents to answers  
✅ **Chunking impact**: Strategy and size affect retrieval  
✅ **Embedding impact**: Model choice affects semantic understanding  
✅ **Retrieval tuning**: top_k optimization  
✅ **LLM comparison**: Different providers and models  
✅ **RAG vs No-RAG**: Clear quality difference  
✅ **Optimal config**: Evidence-based recommendations  

### Key Takeaways:

1. **RAG significantly improves answer quality** for domain-specific questions
2. **Chunking strategy matters**: Recursive with 512 tokens is best default
3. **Embedding model**: all-MiniLM-L6-v2 offers best speed/quality balance
4. **Retrieval**: top_k=5 provides good context without noise
5. **Always compare**: Test different configurations for your use case

### Production Checklist:

- [ ] Use consistent chunking strategy across documents
- [ ] Monitor retrieval scores (low scores = poor matches)
- [ ] Implement feedback loop to improve retrieval
- [ ] Use PgVector for production (scalable)
- [ ] Add caching for frequently asked questions
- [ ] Monitor LLM costs and response times
- [ ] Implement proper error handling
- [ ] Add source citation in responses
- [ ] Version your knowledge base
- [ ] A/B test different configurations

### Next Steps:

1. **Build your own RAG application**
2. **Experiment with your documents**
3. **Fine-tune for your specific domain**
4. **Add conversation history** for chat-like interactions
5. **Deploy to production** with monitoring

### Quick Start Template:

```python
from vectordb import (
    chunk_text, ChunkingConfig, ChunkingStrategy,
    embed_chunks, get_provider, EmbeddingConfig,
    get_backend, store_embeddings, VectorStoreConfig,
    rag_pipeline, RAGConfig, create_embed_function
)

# 1. Chunk
chunks = chunk_text(
    text, "doc.md", 
    ChunkingConfig(strategy=ChunkingStrategy.RECURSIVE, chunk_size=512)
)

# 2. Embed
provider = get_provider("sentence-transformers", EmbeddingConfig())
embeddings = embed_chunks(chunks, provider)

# 3. Store
backend = get_backend("simple", VectorStoreConfig())
store_embeddings(embeddings, backend)

# 4. Query
embed_fn = create_embed_function()
response = rag_pipeline(
    "Your question?", 
    backend, 
    embed_fn, 
    "ollama",
    RAGConfig(top_k=5)
)

print(response.answer)
```